In [1]:
##extractor.py content
from pypdf import PdfReader

## pdf extractor

In [2]:
##the functions begin
##Extract text from notes PDFs, keeping track of which page each piece of text came from
def pdf_pages(pdf_path:str):
    reader = PdfReader(pdf_path)
    return [(i+1, page.extract_text()) for i, page in enumerate(reader.pages)]

## chunker (list of dicta [{page_number, chunk}])

In [3]:
##chunking up the pdf (it remembers its page number)
def chunk_pages(pages, chunk_size=400):
    chunks = []
    for page_num, text in pages:##this is the list of tuples created by our pdf_pages function
        ##check for blank pages (skip if stumbled upon) (pages with images and no text layer get skipped)
        if not text:
            continue
        words = text.split()##list of words in the text of that page number
        for i in range(0, len(words), chunk_size):
            chunks.append({"page": page_num, "text": " ".join(words[i:i+chunk_size])})
    return chunks

In [4]:
##some experimentation
pages = pdf_pages("/Users/tanishqsingh/Desktop/sample_notes.pdf")
chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks in {len(pages)} pages")

3 chunks in 3 pages


In [5]:
pages[:3]

[(1,
  'Operating Systems - Unit 1: Processes\nA process is a program in execution.\nEvery process has a unique Process ID (PID) assigned by the OS.\nProcess states: New, Ready, Running, Waiting, Terminated.\nA context switch saves the state of one process and loads another.\nThe PCB (Process Control Block) stores the process state, program\ncounter, CPU registers, and scheduling information.\nContext switching is pure overhead - no useful work is done during it.\n'),
 (2,
  'CPU Scheduling Algorithms\nFCFS (First Come First Served) is non-preemptive and can cause the\nconvoy effect, where short jobs wait behind a long one.\nSJF (Shortest Job First) gives the minimum average waiting time but\nrequires knowing burst times in advance, which is not practical.\nRound Robin uses a fixed time quantum, typically 10-100 milliseconds.\nIf the quantum is too large, Round Robin degrades into FCFS.\nPriority scheduling can cause starvation, solved by aging.\n'),
 (3,
  'Memory Management\nPaging d

In [7]:
chunks[0]

{'page': 1,
 'text': 'Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.'}

## converting the notes to embeddings and storing them in chroma db

In [6]:
##embeding the chunks that we made in extractor.py
import os
from dotenv import load_dotenv
from google import genai
import chromadb

load_dotenv()##loading the environment vars in the OS

client = genai.Client()
EMBED_M = "gemini-embedding-2"

def embed_text(texts):
    vectors = []
    for text in texts:
        result = client.models.embed_content(
            model=EMBED_M,
            contents=text)
        vectors.append(result.embeddings[0].values)
    return vectors

def build_store(chunks):
    chroma_client = chromadb.PersistentClient()
    notes = chroma_client.get_or_create_collection(name="my_notes")
    vector = embed_text([c["text"] for c in chunks])
    notes.add(
                    ids=[f"c-{i}" for i in range(0,len(chunks))],
                    embeddings=vector,
                    documents=[c["text"] for c in chunks],
                    metadatas=[{"page":c["page"]} for c in chunks],) ##meta data is always list of dicts
    #print(notes.count()) just for checking purposes
    return notes

In [16]:
##some testing
test_result = embed_text(["h"])

In [17]:
test_result

[[-0.0018329112,
  -0.002124854,
  0.027963664,
  -0.035822947,
  -0.00064763863,
  0.0073100794,
  -0.011518925,
  -0.0024478515,
  0.010713634,
  -0.025423393,
  -0.0070703,
  0.015467255,
  0.0051222066,
  0.013139352,
  0.011484409,
  0.001309482,
  0.033579666,
  0.0038086178,
  0.0009656036,
  -0.027254567,
  -0.004491238,
  0.01749954,
  -0.008563499,
  0.012232922,
  -0.012524734,
  -0.018597541,
  -0.008363571,
  0.021689327,
  0.0063870978,
  0.10369742,
  -0.0022621765,
  0.013472584,
  0.021417527,
  -0.0066112177,
  0.02210336,
  -0.0022258395,
  0.0014543389,
  -0.0058239414,
  0.0062591378,
  0.015778486,
  0.012180816,
  -0.0034511928,
  0.0017623607,
  -0.0035350472,
  -0.023337603,
  0.014809829,
  -0.020525612,
  -0.007611531,
  -0.0020985464,
  0.015366201,
  -0.013889007,
  0.003371858,
  -0.0034782551,
  0.015213058,
  0.0065092337,
  -0.0054024076,
  -0.009488893,
  0.0059379255,
  0.007883771,
  -0.0033854041,
  0.014286787,
  -0.006665146,
  0.026512569,
  0.00

In [7]:
##some testing (kill me)
pages = pdf_pages("/Users/tanishqsingh/Desktop/sample_notes.pdf")
chunks = chunk_pages(pages)
coll = build_store(chunks)

## retrieving the contextual chunks from chroma db and using them for answers (with page numbers)

In [8]:
##now working on the retrieval file content which has functions for retrival of document from chroma db
def doc_retrieval(ip_query, notes, n_result=3):
    re_doc = []
    vectors = embed_text([ip_query])##using our vector function
    result = notes.query(
                query_embeddings=vectors,
                n_results=n_result
    )
    for i,j in zip(result['metadatas'][0],result['documents'][0]):
        re_doc.append({'page':i['page'], 'text':j})
    return re_doc

In [9]:
matching = doc_retrieval("what is the convoy effect?",coll)
print(matching)

[{'page': 2, 'text': 'CPU Scheduling Algorithms FCFS (First Come First Served) is non-preemptive and can cause the convoy effect, where short jobs wait behind a long one. SJF (Shortest Job First) gives the minimum average waiting time but requires knowing burst times in advance, which is not practical. Round Robin uses a fixed time quantum, typically 10-100 milliseconds. If the quantum is too large, Round Robin degrades into FCFS. Priority scheduling can cause starvation, solved by aging.'}, {'page': 1, 'text': 'Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.'}, {'page': 3, 'text': 'Memor

## some more testing (end meeeeeeeeee)

In [14]:
a = ("John", "Charles", "Mike")
b = ["Jenny", "Christy", "Monica"]
x = zip(a, b)
for i in x:
    print(i)

('John', 'Jenny')
('Charles', 'Christy')
('Mike', 'Monica')


In [24]:
##even more testing
doc = doc_retrieval("what is the convoy effect?", coll, n_result=2)

In [25]:
for i in enumerate(doc):
    print(i)

(0, {'page': 2, 'text': 'CPU Scheduling Algorithms FCFS (First Come First Served) is non-preemptive and can cause the convoy effect, where short jobs wait behind a long one. SJF (Shortest Job First) gives the minimum average waiting time but requires knowing burst times in advance, which is not practical. Round Robin uses a fixed time quantum, typically 10-100 milliseconds. If the quantum is too large, Round Robin degrades into FCFS. Priority scheduling can cause starvation, solved by aging.'})
(1, {'page': 1, 'text': 'Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.'})


## now we must create a file for asking the llm the query of the user with the context of the retrieved data from chroma db

so this file will follow the following structure-
* an ask function which takes user query (text) as input
* calls the retrieval function to get related context (with page numbers)
* converts it all to a prompt string
* and gives the prompt to the llm
* and returns the output generated

In [37]:
##ask.py
from google import genai
from dotenv import load_dotenv
load_dotenv()##loading all api key into the environment

client = genai.Client()
MODEL = "gemini-3.5-flash-lite"

def fmt_string(chunks_r):
    context = ""
    for i in chunks_r:
        context += f"[page:{i['page']}] {i['text']}\n"
    return context
     
def ask(ip_query, coll):
    retrieved_chunks = doc_retrieval(ip_query, coll)
    fmt_prompt = f"""
    Answer the query strictly from the context below. Cite the page number for every claim.
    If the answer is not in the context, say not in the notes.
    QUERY:
    {ip_query}
    CONTEXT:
    {fmt_string(retrieved_chunks)}
    """
    response = client.interactions.create(
    model=MODEL,
    input=fmt_prompt)
    return response.output_text

In [28]:
##some testing
def fmt_string(chunks_r):
    context = ""
    for i in chunks_r:
        temp = f"[page:{i['page']}] {i['text']}\n"
        context += temp
    return context

test_f_s = fmt_string(doc)

In [30]:
print(test_f_s)

[page:2] CPU Scheduling Algorithms FCFS (First Come First Served) is non-preemptive and can cause the convoy effect, where short jobs wait behind a long one. SJF (Shortest Job First) gives the minimum average waiting time but requires knowing burst times in advance, which is not practical. Round Robin uses a fixed time quantum, typically 10-100 milliseconds. If the quantum is too large, Round Robin degrades into FCFS. Priority scheduling can cause starvation, solved by aging.
[page:1] Operating Systems - Unit 1: Processes A process is a program in execution. Every process has a unique Process ID (PID) assigned by the OS. Process states: New, Ready, Running, Waiting, Terminated. A context switch saves the state of one process and loads another. The PCB (Process Control Block) stores the process state, program counter, CPU registers, and scheduling information. Context switching is pure overhead - no useful work is done during it.



In [33]:
##maybe the final testing 
res = ask("what is the convoy effect?", coll)

In [34]:
res

'The convoy effect is a phenomenon where short jobs wait behind a long one [page: 2].'

In [38]:
res1 = ask("who is the indain primeministee?", coll)

In [39]:
res1

'Not in the notes.'